# Train and export the household power model

Run this notebook from the project root. The dataset is expected at `dataset/household_power_consumption.txt`. It creates `models/model.joblib` and `models/model_metadata.json`.

Voltage and Global_intensity are used only as lagged inputs. The rolling mean contains completed hours before the prediction time.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'dataset' / 'household_power_consumption.txt'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

## Load and resample

The file uses semicolons and `?` for missing values. Aggregate numeric measurements to hourly means and interpolate missing hourly values.

In [2]:
df = pd.read_csv(DATASET_PATH, sep=';', na_values=['?'], low_memory=False)
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)
df = df.set_index('Datetime').drop(columns=['Date', 'Time'])

for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df_hourly = df.resample('h').mean()
df_hourly = df_hourly.interpolate(method='linear')
df_hourly.head()

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:00:00,4.222889,0.229000,234.643889,18.100000,0.0,0.527778,16.861111
2006-12-16 18:00:00,3.632200,0.080033,234.580167,15.600000,0.0,6.716667,16.866667
2006-12-16 19:00:00,3.400233,0.085233,233.232500,14.503333,0.0,1.433333,16.683333
2006-12-16 20:00:00,3.268567,0.075100,234.071500,13.916667,0.0,0.000000,16.783333
2006-12-16 21:00:00,3.056467,0.076667,237.158667,13.046667,0.0,0.416667,17.216667


## Engineer leakage-safe features

Voltage and intensity enter only as lagged values. Shift the 24-hour rolling mean so it excludes the target hour.

In [3]:
TARGET = 'Global_active_power'
feature_columns = [
    'power_lag_1h',
    'power_lag_24h',
    'voltage_lag_1h',
    'intensity_lag_1h',
    'rolling_mean_24h',
]

df_hourly['power_lag_1h'] = df_hourly[TARGET].shift(1)
df_hourly['power_lag_24h'] = df_hourly[TARGET].shift(24)
df_hourly['voltage_lag_1h'] = df_hourly['Voltage'].shift(1)
df_hourly['intensity_lag_1h'] = df_hourly['Global_intensity'].shift(1)
df_hourly['rolling_mean_24h'] = (
    df_hourly[TARGET].rolling(window=24).mean().shift(1)
)
df_hourly = df_hourly.dropna(subset=feature_columns + [TARGET])

X = df_hourly[feature_columns]
y = df_hourly[TARGET]
print(f'Rows available: {len(df_hourly):,}')
print(f'Features in model order: {feature_columns}')

Rows available: 34,565
Features in model order: ['power_lag_1h', 'power_lag_24h', 'voltage_lag_1h', 'intensity_lag_1h', 'rolling_mean_24h']


## Chronological split, train, and evaluate

Use the first 80% of rows for training and the final 20% for evaluation, without shuffling.

In [4]:
train_size = int(len(df_hourly) * 0.8)
if train_size < 1 or train_size >= len(df_hourly):
    raise ValueError('Not enough hourly rows for an 80/20 chronological split.')

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE: {mae:.4f}')

Test RMSE: 0.5148
Test MAE: 0.3629


## Export the model and feature metadata

Save the trained model and exact feature names in training order.

In [5]:
model_path = MODELS_DIR / 'model.joblib'
joblib.dump(model, model_path)

metadata = {
    'feature_names': feature_columns,
    'target': TARGET,
}
metadata_path = MODELS_DIR / 'model_metadata.json'
with metadata_path.open('w', encoding='utf-8') as metadata_file:
    json.dump(metadata, metadata_file, indent=4)
    metadata_file.write('\n')

print(f'Model saved to {model_path}')
print(f'Metadata saved to {metadata_path}')

Model saved to c:\Users\upremier\Documents\new procect\Formative1_Pipeline_For_Time_Series\models\model.joblib
Metadata saved to c:\Users\upremier\Documents\new procect\Formative1_Pipeline_For_Time_Series\models\model_metadata.json
